# Prototype: tabular foundation model against the tuned baseline

**This is an exploratory prototype and its numbers are not project results.** It uses its own split and its own preprocessing, so its figures are not comparable with the four measured tracks. The published numbers come only from `reports/track_comparison.json`, produced by `scripts/run_comparison.py`.

The prototype is kept because its finding is worth recording: on this dataset the foundation model was not distinguishable from the tuned gradient booster, with every paired interval spanning zero. Integrating it as a fifth track is roadmap item 3.8.

# Environment Check

In [1]:
import os
import platform
import shutil
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import sklearn

from src.data.registry import GERMAN_CREDIT

# The privileged value is a per-dataset registry entry, not a global constant:
# the disadvantaged group is women on German Credit and men on Taiwan.
SEX = next(a for a in GERMAN_CREDIT.protected if a.column == 'gender')
from src.paths import PROCESSED_DATA_DIR

print("python      :", sys.version.split()[0], "|", platform.machine())
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("sklearn     :", sklearn.__version__)
print("repo root   :", REPO_ROOT.name)
print("data exists :", GERMAN_CREDIT.path.exists())

free_gb = shutil.disk_usage(REPO_ROOT).free / 1024**3
print(f"free disk   : {free_gb:.1f} GB")

try:
    import torch
    print("torch       :", torch.__version__, "| cuda:", torch.cuda.is_available())
except ImportError:
    print("torch       : not installed")

print("cpu count   :", os.cpu_count())

python      : 3.12.10 | AMD64
numpy       : 2.2.0
pandas      : 2.2.3
sklearn     : 1.6.0
repo root   : fairness-credit-risk
data exists : True
free disk   : 56.3 GB
torch       : not installed
cpu count   : 16


# Data Preprocessing & Feature Framing

In [2]:
df = pd.read_csv(GERMAN_CREDIT.path)

TARGET = "credit_risk"

NUMERIC = [
    "duration", "amount", "age", "installment_rate",
    "present_residence", "number_credits", "people_liable",
]

CATEGORICAL = [
    "status", "credit_history", "purpose", "savings", "employment_duration",
    "other_debtors", "property", "other_installment_plans", "housing",
    "job", "telephone",
]

EXCLUDED = ["personal_status_sex", "gender", "foreign_worker", "age_group"]
FEATURES = NUMERIC + CATEGORICAL

assert set(FEATURES + EXCLUDED + [TARGET]) == set(df.columns), (
    set(df.columns) ^ set(FEATURES + EXCLUDED + [TARGET])
)
assert len(FEATURES) == 18

print("features        :", len(FEATURES), f"({len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical)")
print("age in features :", "age" in FEATURES)
print("sex proxy in    :", "personal_status_sex" in FEATURES)
print("target balance  :", df[TARGET].value_counts().to_dict())
print("gender balance  :", df["gender"].value_counts().to_dict())

X_num = df[FEATURES].copy()

X_cat = df[FEATURES].copy()
for col in CATEGORICAL:
    X_cat[col] = (col + "_" + X_cat[col].astype(str)).astype("category")

y = df[TARGET].to_numpy()
protected = df[["gender"]].copy()

print("\nsklearn dtypes :", X_num.dtypes.value_counts().to_dict())
print("tabfm dtypes   :", X_cat.dtypes.value_counts().to_dict())
print("\nsample data:")
print(X_cat[["status", "duration", "purpose", "amount", "age"]].head(3))

features        : 18 (7 numeric, 11 categorical)
age in features : True
sex proxy in    : False
target balance  : {0: 700, 1: 300}
gender balance  : {1: 690, 0: 310}

sklearn dtypes : {dtype('int64'): 18}
tabfm dtypes   : {dtype('int64'): 7, CategoricalDtype(categories=['status_0', 'status_1', 'status_2', 'status_3'], ordered=False, categories_dtype=object): 1, CategoricalDtype(categories=['credit_history_0', 'credit_history_1', 'credit_history_2',
                  'credit_history_3', 'credit_history_4'],
, ordered=False, categories_dtype=object): 1, CategoricalDtype(categories=['purpose_0', 'purpose_1', 'purpose_2', 'purpose_3',
                  'purpose_4', 'purpose_5', 'purpose_6', 'purpose_7',
                  'purpose_8', 'purpose_9'],
, ordered=False, categories_dtype=object): 1, CategoricalDtype(categories=['savings_0', 'savings_1', 'savings_2', 'savings_3',
                  'savings_4'],
, ordered=False, categories_dtype=object): 1, CategoricalDtype(categories=['employment_

# Stratified Data Splitting

In [3]:
from sklearn.model_selection import train_test_split

SEED = 42  # matches config.RANDOM_STATE

strata = df[TARGET].astype(str) + "_" + df["gender"].astype(str)
idx_all = np.arange(len(df))

idx_fit, idx_test = train_test_split(
    idx_all, test_size=0.20, stratify=strata, random_state=SEED
)
idx_train, idx_calib = train_test_split(
    idx_fit, test_size=0.25, stratify=strata[idx_fit], random_state=SEED
)

splits = {"train": idx_train, "calib": idx_calib, "test": idx_test}

assert len(idx_train) == 600 and len(idx_calib) == 200 and len(idx_test) == 200
assert set(idx_train) & set(idx_calib) == set()
assert set(idx_train) & set(idx_test) == set()
assert set(idx_calib) & set(idx_test) == set()
assert len(set(idx_train) | set(idx_calib) | set(idx_test)) == 1000

for name, idx in splits.items():
    bad_rate = y[idx].mean()
    male_share = protected["gender"].to_numpy()[idx].mean()
    print(f"{name:6}: n={len(idx):4d}  bad_rate={bad_rate:.4f}  male_share={male_share:.4f}")

print("\nfull  : bad_rate=%.4f  male_share=%.4f" % (y.mean(), protected["gender"].mean()))

train : n= 600  bad_rate=0.3000  male_share=0.6900
calib : n= 200  bad_rate=0.3000  male_share=0.6900
test  : n= 200  bad_rate=0.3000  male_share=0.6900

full  : bad_rate=0.3000  male_share=0.6900


# Fairness Evaluation Metrics

In [4]:
from dataclasses import dataclass

FAVORABLE = GERMAN_CREDIT.favorable_label          # 0 = good credit
PRIV = SEX.privileged_value                        # 1 = male
UNPRIV = SEX.unprivileged_value                    # 0 = female


@dataclass(frozen=True)
class GroupRates:
    selection_rate: float
    tpr: float
    fpr: float
    n: int


def _group_rates(y_true, y_pred, mask) -> GroupRates:
    yt, yp = np.asarray(y_true)[mask], np.asarray(y_pred)[mask]
    fav_true = yt == FAVORABLE
    unfav_true = ~fav_true
    return GroupRates(
        selection_rate=float(np.mean(yp == FAVORABLE)) if len(yp) else float("nan"),
        tpr=float(np.mean(yp[fav_true] == FAVORABLE)) if fav_true.any() else float("nan"),
        fpr=float(np.mean(yp[unfav_true] == FAVORABLE)) if unfav_true.any() else float("nan"),
        n=int(len(yp)),
    )


def fairness_metrics(y_true, y_pred, group) -> dict:
    group = np.asarray(group)
    priv = _group_rates(y_true, y_pred, group == PRIV)
    unpriv = _group_rates(y_true, y_pred, group == UNPRIV)

    di = unpriv.selection_rate / priv.selection_rate if priv.selection_rate > 0 else float("inf")
    return {
        "disparate_impact": di,
        "statistical_parity_difference": unpriv.selection_rate - priv.selection_rate,
        "equal_opportunity_difference": unpriv.tpr - priv.tpr,
        "equalized_odds_difference": max(abs(unpriv.tpr - priv.tpr), abs(unpriv.fpr - priv.fpr)),
        "selection_rate_privileged": priv.selection_rate,
        "selection_rate_unprivileged": unpriv.selection_rate,
        "n_privileged": priv.n,
        "n_unprivileged": unpriv.n,
    }


# Verification 1: Hand-derived DI and SPD
yt = np.array([0, 0, 0, 0, 0, 0, 0, 0])
yp = np.array([0, 0, 0, 1, 0, 0, 1, 1])
g = np.array([1, 1, 1, 1, 0, 0, 0, 0])
m = fairness_metrics(yt, yp, g)
assert abs(m["disparate_impact"] - 2 / 3) < 1e-9, m["disparate_impact"]
assert abs(m["statistical_parity_difference"] - (-0.25)) < 1e-9
print("verification 1 (hand-derived metrics): pass")

# Verification 2: Predictions sensitivity
yp_all_good = np.zeros(8, dtype=int)
m_all_good = fairness_metrics(yt, yp_all_good, g)
assert m_all_good["disparate_impact"] == 1.0
assert m_all_good["statistical_parity_difference"] == 0.0
assert m_all_good["disparate_impact"] != m["disparate_impact"]
print("verification 2 (prediction dependency): pass")

# Reference check on test ground truth
label_as_pred = fairness_metrics(
    y[idx_test], y[idx_test], protected["gender"].to_numpy()[idx_test]
)
print(f"\nground-truth label DI on test set  : {label_as_pred['disparate_impact']:.4f}")
print(f"ground-truth label SPD on test set : {label_as_pred['statistical_parity_difference']:.4f}")

verification 1 (hand-derived metrics): pass
verification 2 (prediction dependency): pass

ground-truth label DI on test set  : 0.8903
ground-truth label SPD on test set : -0.0795


# Sample Reweighing Weights

In [5]:
def reweighing_weights(y_true, group) -> np.ndarray:
    """Kamiran & Calders (2012) reweighing."""
    y_true, group = np.asarray(y_true), np.asarray(group)
    n = len(y_true)
    weights = np.ones(n, dtype=float)
    for a in np.unique(group):
        for label in np.unique(y_true):
            cell = (group == a) & (y_true == label)
            n_cell = cell.sum()
            if n_cell == 0:
                continue
            weights[cell] = ((group == a).sum() * (y_true == label).sum()) / (n * n_cell)
    return weights


# Hand-derived check
yt_w = np.array([0, 0, 0, 1, 0, 1, 1, 1])
g_w = np.array([1, 1, 1, 1, 0, 0, 0, 0])
w = reweighing_weights(yt_w, g_w)
assert abs(w[0] - 2 / 3) < 1e-9, w[0]
assert abs(w[3] - 2.0) < 1e-9, w[3]
assert abs(w[4] - 2.0) < 1e-9, w[4]
assert abs(w.sum() - len(yt_w)) < 1e-9, w.sum()
print("reweighing self-test: pass")

w_train = reweighing_weights(y[idx_train], protected["gender"].to_numpy()[idx_train])
print(f"train weights: min={w_train.min():.4f} max={w_train.max():.4f} mean={w_train.mean():.4f}")
print("distinct weights:", np.round(np.unique(w_train), 4))

reweighing self-test: pass
train weights: min=0.8585 max=1.0800 mean=1.0000
distinct weights: [0.8585 0.9692 1.076  1.08  ]


# Baseline Tracks (T0 & T1)

In [6]:
import time

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

POSITIVE_CLASS_INDEX = 1


def make_preprocessor() -> ColumnTransformer:
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
    ])


def performance_metrics(y_true, y_pred, y_proba) -> dict:
    return {
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
    }


def run_sklearn_track(name, sample_weight=None) -> dict:
    pre = make_preprocessor()
    Xtr = pre.fit_transform(X_num.iloc[idx_train])
    Xte = pre.transform(X_num.iloc[idx_test])

    model = RandomForestClassifier(
        n_estimators=300, min_samples_leaf=4,
        class_weight="balanced", random_state=SEED, n_jobs=-1,
    )
    t0 = time.perf_counter()
    model.fit(Xtr, y[idx_train], sample_weight=sample_weight)
    fit_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    proba = model.predict_proba(Xte)[:, POSITIVE_CLASS_INDEX]
    predict_s = time.perf_counter() - t0

    pred = (proba >= 0.5).astype(int)
    g_test = protected["gender"].to_numpy()[idx_test]

    return {
        "track": name,
        "n_features_encoded": Xtr.shape[1],
        "fit_seconds": round(fit_s, 3),
        "predict_seconds": round(predict_s, 4),
        **performance_metrics(y[idx_test], pred, proba),
        **fairness_metrics(y[idx_test], pred, g_test),
    }


results = {}
results["T0_unmitigated"] = run_sklearn_track("T0_unmitigated")
results["T1_reweighing"] = run_sklearn_track("T1_reweighing", sample_weight=w_train)

summary = pd.DataFrame(results).T
cols = ["roc_auc", "pr_auc", "balanced_accuracy", "f1", "recall",
        "disparate_impact", "statistical_parity_difference",
        "equal_opportunity_difference", "equalized_odds_difference",
        "selection_rate_privileged", "selection_rate_unprivileged"]
print(summary[cols].astype(float).round(4).to_string())
print("\nencoded feature width:", results["T0_unmitigated"]["n_features_encoded"])
print("predict latency (s)  :", results["T0_unmitigated"]["predict_seconds"])

                roc_auc  pr_auc  balanced_accuracy      f1  recall  disparate_impact  statistical_parity_difference  equal_opportunity_difference  equalized_odds_difference  selection_rate_privileged  selection_rate_unprivileged
T0_unmitigated   0.8269  0.6526             0.7369  0.6299  0.6667            0.7052                        -0.2158                        -0.185                      0.185                     0.7319                       0.5161
T1_reweighing    0.8195  0.6597             0.7131  0.5984  0.6333            0.7345                        -0.1924                        -0.165                      0.165                     0.7246                       0.5323

encoded feature width: 55
predict latency (s)  : 0.0369


# Bootstrap Confidence Intervals (T0 vs T1)

In [7]:
def bootstrap_metrics(y_true, y_pred, y_proba, group, n_bootstraps=1000, seed=SEED):
    rng = np.random.RandomState(seed)
    metrics_list = []
    n = len(y_true)
    for _ in range(n_bootstraps):
        idx = rng.choice(n, size=n, replace=True)
        m = {
            **performance_metrics(y_true[idx], y_pred[idx], y_proba[idx]),
            **fairness_metrics(y_true[idx], y_pred[idx], group[idx]),
        }
        metrics_list.append(m)
    boot_df = pd.DataFrame(metrics_list)
    return pd.DataFrame({
        "mean": boot_df.mean(),
        "std": boot_df.std(),
        "ci_2.5%": boot_df.quantile(0.025),
        "ci_97.5%": boot_df.quantile(0.975),
    })

g_test = protected["gender"].to_numpy()[idx_test]
pre = make_preprocessor()
Xtr = pre.fit_transform(X_num.iloc[idx_train])
Xte = pre.transform(X_num.iloc[idx_test])

model_t0 = RandomForestClassifier(n_estimators=300, min_samples_leaf=4, class_weight="balanced", random_state=SEED, n_jobs=-1)
model_t0.fit(Xtr, y[idx_train])
proba_t0 = model_t0.predict_proba(Xte)[:, POSITIVE_CLASS_INDEX]
pred_t0 = (proba_t0 >= 0.5).astype(int)

model_t1 = RandomForestClassifier(n_estimators=300, min_samples_leaf=4, class_weight="balanced", random_state=SEED, n_jobs=-1)
model_t1.fit(Xtr, y[idx_train], sample_weight=w_train)
proba_t1 = model_t1.predict_proba(Xte)[:, POSITIVE_CLASS_INDEX]
pred_t1 = (proba_t1 >= 0.5).astype(int)

ci_t0 = bootstrap_metrics(y[idx_test], pred_t0, proba_t0, g_test)
ci_t1 = bootstrap_metrics(y[idx_test], pred_t1, proba_t1, g_test)

print("T0 Bootstrap CIs:")
print(ci_t0.loc[["roc_auc", "disparate_impact", "statistical_parity_difference"]].round(4))
print("\nT1 Bootstrap CIs:")
print(ci_t1.loc[["roc_auc", "disparate_impact", "statistical_parity_difference"]].round(4))
print(f"\nPrediction flips between T0 and T1: {(pred_t0 != pred_t1).sum()} / {len(pred_t0)}")

T0 Bootstrap CIs:
                                 mean     std  ci_2.5%  ci_97.5%
roc_auc                        0.8290  0.0297   0.7690    0.8843
disparate_impact               0.7082  0.0960   0.5268    0.8955
statistical_parity_difference -0.2144  0.0745  -0.3582   -0.0722

T1 Bootstrap CIs:
                                 mean     std  ci_2.5%  ci_97.5%
roc_auc                        0.8214  0.0305   0.7616    0.8782
disparate_impact               0.7371  0.0963   0.5515    0.9288
statistical_parity_difference -0.1914  0.0736  -0.3346   -0.0489

Prediction flips between T0 and T1: 4 / 200


# Export Portable Exchange File

In [8]:
EXCHANGE = REPO_ROOT / "notebooks" / "exchange"
EXCHANGE.mkdir(parents=True, exist_ok=True)

split_label = pd.Series("unassigned", index=df.index, dtype=object)
split_label.iloc[idx_train] = "train"
split_label.iloc[idx_calib] = "calib"
split_label.iloc[idx_test] = "test"
assert (split_label != "unassigned").all()

export = X_cat.copy()
for col in CATEGORICAL:
    export[col] = export[col].astype(str)
export["__row_id"] = np.arange(len(df))
export["__y"] = y
export["__gender"] = protected["gender"].to_numpy()
export["__split"] = split_label.to_numpy()

out_path = EXCHANGE / "tabfm_input.csv"
export.to_csv(out_path, index=False)

print("wrote:", out_path.relative_to(REPO_ROOT))
print("rows :", len(export), "| split counts:", export["__split"].value_counts().to_dict())
print("cols :", [c for c in export.columns if c.startswith("__")])
print("first 3 test row ids:", export.loc[export["__split"] == "test", "__row_id"].head(3).tolist())

wrote: notebooks\exchange\tabfm_input.csv
rows : 1000 | split counts: {'train': 600, 'calib': 200, 'test': 200}
cols : ['__row_id', '__y', '__gender', '__split']
first 3 test row ids: [2, 14, 17]


# Final Track Comparison (T0, T1, T4)

In [9]:
import json
import pandas as pd

N_BOOT = 1000
METRICS = [
    "roc_auc", "pr_auc", "balanced_accuracy", "f1",
    "disparate_impact", "statistical_parity_difference",
    "equal_opportunity_difference", "equalized_odds_difference"
]

def all_metrics(y_true, y_pred, y_proba, group):
    return {
        **performance_metrics(y_true, y_pred, y_proba),
        **fairness_metrics(y_true, y_pred, group),
    }

g_test = protected["gender"].to_numpy()[idx_test]
yt = y[idx_test]

pre = make_preprocessor()
Xtr = pre.fit_transform(X_num.iloc[idx_train])
Xte = pre.transform(X_num.iloc[idx_test])

model_t0 = RandomForestClassifier(n_estimators=300, min_samples_leaf=4, class_weight="balanced", random_state=SEED, n_jobs=-1)
model_t0.fit(Xtr, y[idx_train])
proba_t0 = model_t0.predict_proba(Xte)[:, POSITIVE_CLASS_INDEX]
pred_t0 = (proba_t0 >= 0.5).astype(int)

model_t1 = RandomForestClassifier(n_estimators=300, min_samples_leaf=4, class_weight="balanced", random_state=SEED, n_jobs=-1)
model_t1.fit(Xtr, y[idx_train], sample_weight=w_train)
proba_t1 = model_t1.predict_proba(Xte)[:, POSITIVE_CLASS_INDEX]
pred_t1 = (proba_t1 >= 0.5).astype(int)

# Load TabFM predictions and realign to local test order
pred_df = pd.read_csv(EXCHANGE / "tabfm_predictions_german.csv")
meta_t4 = json.loads((EXCHANGE / "tabfm_meta_german.json").read_text())

proba_by_id = pred_df.set_index("__row_id")["proba_default"]
assert set(proba_by_id.index) == set(idx_test), "TabFM scored a different set of rows"
assert len(proba_by_id) == len(idx_test)

proba_t4 = proba_by_id.loc[idx_test].to_numpy()
pred_t4 = (proba_t4 >= 0.5).astype(int)

tracks = {
    "T0_unmitigated": (pred_t0, proba_t0),
    "T1_reweighing": (pred_t1, proba_t1),
    "T4_tabfm": (pred_t4, proba_t4),
}

rng = np.random.default_rng(SEED)
replicates = [rng.choice(len(yt), size=len(yt), replace=True) for _ in range(N_BOOT)]

point, intervals = {}, {}
for name, (pred, proba) in tracks.items():
    point[name] = all_metrics(yt, pred, proba, g_test)
    draws = [all_metrics(yt[b], pred[b], proba[b], g_test[b]) for b in replicates]
    intervals[name] = {
        k: tuple(np.nanpercentile([d[k] for d in draws], [2.5, 97.5])) for k in METRICS
    }

print("%-30s %-26s %-26s %-26s" % ("metric", "T0_unmitigated", "T1_reweighing", "T4_tabfm"))
for k in METRICS:
    cells_out = []
    for name in tracks:
        lo, hi = intervals[name][k]
        cells_out.append(f"{point[name][k]:+.4f} [{lo:+.4f},{hi:+.4f}]")
    print("%-30s %-26s %-26s %-26s" % (k, *cells_out))

print("\npaired difference vs T0 (95% CI; excludes 0 => distinguishable)")
for name in ("T1_reweighing", "T4_tabfm"):
    pred, proba = tracks[name]
    flips = int((pred != pred_t0).sum())
    print(f"\n  {name}  ({flips}/{len(pred)} prediction flips vs T0)")
    for k in METRICS:
        diffs = [all_metrics(yt[b], pred[b], proba[b], g_test[b])[k]
                 - all_metrics(yt[b], pred_t0[b], proba_t0[b], g_test[b])[k]
                 for b in replicates]
        lo, hi = np.nanpercentile(diffs, [2.5, 97.5])
        verdict = "distinguishable" if (lo > 0 or hi < 0) else "not distinguishable"
        print(f"    {k:30} {np.nanmean(diffs):+.4f} [{lo:+.4f},{hi:+.4f}]  {verdict}")

print("\ncalibration (mean predicted P(default) vs actual %.4f)" % yt.mean())
for name, (_, proba) in tracks.items():
    brier = float(np.mean((proba - yt) ** 2))
    print(f"  {name:18} mean_proba={proba.mean():.4f}  brier={brier:.4f}")

print("\nlatency per row (seconds)")
print(f"  T0/T1 random forest : {results['T0_unmitigated']['predict_seconds'] / 200:.6f}")
print(f"  T4 tabfm (T4 GPU)   : {meta_t4['predict_seconds_per_row']:.6f}")
print(f"  ratio               : {meta_t4['predict_seconds_per_row'] / (results['T0_unmitigated']['predict_seconds'] / 200):.0f}x slower")

artifact = {
    "dataset": "german_credit",
    "n_test": int(len(idx_test)),
    "n_privileged_test": int((g_test == PRIV).sum()),
    "n_unprivileged_test": int((g_test == UNPRIV).sum()),
    "n_bootstrap": N_BOOT,
    "tracks": {
        name: {
            "point": {k: float(point[name][k]) for k in METRICS},
            "ci95": {k: [float(v) for v in intervals[name][k]] for k in METRICS},
        }
        for name in tracks
    },
    "tabfm_meta": meta_t4,
}
reports_dir = REPO_ROOT / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
(reports_dir / "prototype_comparison_german.json").write_text(json.dumps(artifact, indent=2))
print("\nwrote reports/prototype_comparison_german.json")

metric                         T0_unmitigated             T1_reweighing              T4_tabfm                  
roc_auc                        +0.8269 [+0.7686,+0.8793]  +0.8195 [+0.7623,+0.8728]  +0.8435 [+0.7929,+0.8914] 
pr_auc                         +0.6526 [+0.5263,+0.7732]  +0.6597 [+0.5394,+0.7688]  +0.6968 [+0.5817,+0.8006] 
balanced_accuracy              +0.7369 [+0.6657,+0.8006]  +0.7131 [+0.6404,+0.7769]  +0.7155 [+0.6485,+0.7829] 
f1                             +0.6299 [+0.5234,+0.7170]  +0.5984 [+0.4827,+0.6862]  +0.6018 [+0.4915,+0.6990] 
disparate_impact               +0.7052 [+0.5360,+0.9085]  +0.7345 [+0.5602,+0.9321]  +0.8321 [+0.6818,+1.0043] 
statistical_parity_difference  -0.2158 [-0.3556,-0.0651]  -0.1924 [-0.3271,-0.0469]  -0.1302 [-0.2568,+0.0031] 
equal_opportunity_difference   -0.1850 [-0.3386,-0.0245]  -0.1650 [-0.3195,-0.0052]  -0.0550 [-0.1819,+0.0739] 
equalized_odds_difference      +0.1850 [+0.0809,+0.4001]  +0.1650 [+0.0628,+0.3902]  +0.1818 [+0.0324,+0

    roc_auc                        -0.0075 [-0.0180,+0.0024]  not distinguishable


    pr_auc                         +0.0054 [-0.0158,+0.0342]  not distinguishable


    balanced_accuracy              -0.0245 [-0.0531,-0.0035]  distinguishable


    f1                             -0.0325 [-0.0724,-0.0047]  distinguishable


...

    f1                             -0.0270 [-0.1057,+0.0503]  not distinguishable


    disparate_impact               +0.1289 [+0.0182,+0.2501]  distinguishable


    statistical_parity_difference  +0.0876 [+0.0024,+0.1837]  distinguishable


    equal_opportunity_difference   +0.1301 [+0.0227,+0.2439]  distinguishable


    equalized_odds_difference      -0.0385 [-0.2034,+0.1511]  not distinguishable

calibration (mean predicted P(default) vs actual 0.3000)
  T0_unmitigated     mean_proba=0.4067  brier=0.1673
  T1_reweighing      mean_proba=0.4047  brier=0.1678
  T4_tabfm           mean_proba=0.3105  brier=0.1466

latency per row (seconds)
  T0/T1 random forest : 0.000185
  T4 tabfm (T4 GPU)   : 0.266200
  ratio               : 1443x slower

wrote reports/prototype_comparison_german.json


In [10]:
def sel_rate(pred):
    return float(np.mean(pred == FAVORABLE))

print("global selection rate (fraction approved) at threshold 0.5")
for name, (pred, _) in tracks.items():
    print(f"  {name:18} {sel_rate(pred):.4f}")

# Match every track to T0's approval rate. Predicted favorable when proba < t,
# so the threshold achieving selection rate q is the q-th quantile of the scores.
target = sel_rate(pred_t0)
print(f"\nmatching all tracks to T0 selection rate = {target:.4f}")

matched = {}
for name, (_, proba) in tracks.items():
    t = float(np.quantile(proba, target))
    pred_m = (proba >= t).astype(int)
    matched[name] = (pred_m, proba, t)
    print(f"  {name:18} threshold={t:.4f}  achieved_sr={sel_rate(pred_m):.4f}")

FAIR = ["disparate_impact", "statistical_parity_difference",
        "equal_opportunity_difference", "equalized_odds_difference"]

print("\nfairness at MATCHED selection rate")
print("%-30s %-26s %-26s %-26s" % ("metric", *tracks.keys()))
for k in FAIR:
    row = []
    for name in tracks:
        pred_m, proba, _ = matched[name]
        draws = [fairness_metrics(yt[b], pred_m[b], g_test[b])[k] for b in replicates]
        lo, hi = np.nanpercentile(draws, [2.5, 97.5])
        pt = fairness_metrics(yt, pred_m, g_test)[k]
        row.append(f"{pt:+.4f} [{lo:+.4f},{hi:+.4f}]")
    print("%-30s %-26s %-26s %-26s" % (k, *row))

print("\npaired difference vs T0 at matched selection rate")
pred_ref = matched["T0_unmitigated"][0]
for name in ("T1_reweighing", "T4_tabfm"):
    pred_m = matched[name][0]
    print(f"\n  {name}  ({int((pred_m != pred_ref).sum())}/{len(pred_m)} flips vs T0)")
    for k in FAIR:
        diffs = [fairness_metrics(yt[b], pred_m[b], g_test[b])[k]
                 - fairness_metrics(yt[b], pred_ref[b], g_test[b])[k] for b in replicates]
        lo, hi = np.nanpercentile(diffs, [2.5, 97.5])
        verdict = "distinguishable" if (lo > 0 or hi < 0) else "not distinguishable"
        print(f"    {k:30} {np.nanmean(diffs):+.4f} [{lo:+.4f},{hi:+.4f}]  {verdict}")

print("\nBrier score with 95% CI (lower is better; unaffected by threshold)")
for name, (_, proba) in tracks.items():
    draws = [float(np.mean((proba[b] - yt[b]) ** 2)) for b in replicates]
    lo, hi = np.nanpercentile(draws, [2.5, 97.5])
    print(f"  {name:18} {np.mean((proba - yt) ** 2):.4f} [{lo:.4f},{hi:.4f}]")

print("\npaired Brier difference vs T0 (negative = TabFM better calibrated)")
for name in ("T1_reweighing", "T4_tabfm"):
      proba = tracks[name][1]
      diffs = [float(np.mean((proba[b] - yt[b]) ** 2) - np.mean((proba_t0[b] - yt[b]) ** 2))
               for b in replicates]
      lo, hi = np.nanpercentile(diffs, [2.5, 97.5])
      verdict = "distinguishable" if (lo > 0 or hi < 0) else "not distinguishable"
      print(f"  {name:18} {np.mean(diffs):+.4f} [{lo:+.4f},{hi:+.4f}]  {verdict}")

global selection rate (fraction approved) at threshold 0.5
  T0_unmitigated     0.6650
  T1_reweighing      0.6650
  T4_tabfm           0.7350

matching all tracks to T0 selection rate = 0.6650
  T0_unmitigated     threshold=0.4985  achieved_sr=0.6650
  T1_reweighing      threshold=0.4963  achieved_sr=0.6650
  T4_tabfm           threshold=0.4101  achieved_sr=0.6650

fairness at MATCHED selection rate
metric                         T0_unmitigated             T1_reweighing              T4_tabfm                  
disparate_impact               +0.7052 [+0.5360,+0.9085]  +0.7345 [+0.5602,+0.9321]  +0.7949 [+0.6104,+0.9981] 


statistical_parity_difference  -0.2158 [-0.3556,-0.0651]  -0.1924 [-0.3271,-0.0469]  -0.1456 [-0.2866,-0.0012] 
equal_opportunity_difference   -0.1850 [-0.3386,-0.0245]  -0.1650 [-0.3195,-0.0052]  -0.0900 [-0.2489,+0.0749] 


equalized_odds_difference      +0.1850 [+0.0809,+0.4001]  +0.1650 [+0.0628,+0.3902]  +0.1411 [+0.0396,+0.3593] 

paired difference vs T0 at matched selection rate

  T1_reweighing  (4/200 flips vs T0)
    disparate_impact               +0.0309 [-0.0088,+0.0924]  not distinguishable
    statistical_parity_difference  +0.0246 [-0.0079,+0.0714]  not distinguishable


    equal_opportunity_difference   +0.0211 [+0.0000,+0.0538]  not distinguishable
    equalized_odds_difference      -0.0171 [-0.0910,+0.0652]  not distinguishable

  T4_tabfm  (12/200 flips vs T0)
    disparate_impact               +0.0929 [+0.0174,+0.1982]  distinguishable


    statistical_parity_difference  +0.0723 [+0.0098,+0.1502]  distinguishable
    equal_opportunity_difference   +0.0976 [+0.0107,+0.2028]  distinguishable
    equalized_odds_difference      -0.0579 [-0.1636,+0.0571]  not distinguishable

Brier score with 95% CI (lower is better; unaffected by threshold)
  T0_unmitigated     0.1673 [0.1488,0.1865]
  T1_reweighing      0.1678 [0.1490,0.1873]
  T4_tabfm           0.1466 [0.1215,0.1747]

paired Brier difference vs T0 (negative = TabFM better calibrated)
  T1_reweighing      +0.0005 [-0.0020,+0.0030]  not distinguishable
  T4_tabfm           -0.0207 [-0.0330,-0.0072]  distinguishable
